# Nicolay Corpus Re-Indexing Notebook (v1.4)

**Project:** Nicolay — A Transparent RAG System for Lincoln Corpus Exploration  
**Article:** *Hutchinson, Daniel. "Nicolay: Transparent Retrieval-Augmented Generation for Historical Corpus Exploration." Digital Humanities Quarterly (forthcoming).*  
**GitHub:** [github.com/Dr-Hutchinson/nicolay](https://github.com/Dr-Hutchinson/nicolay)

---

## About This Notebook

This notebook builds the searchable index that powers Nicolay's retrieval pipeline. It ingests raw Lincoln speech texts and Lincoln-Douglas Debate transcripts, chunks them into semantically coherent passages, generates LLM-assisted metadata (summaries and keywords) for each chunk, and produces Ada-002 embeddings for semantic search. The two output files — `lincoln_speech_corpus.json` and `lincoln_index_embedded.csv` — are the runtime data store and embedding index used by the Nicolay application.

This notebook is one of a series of iterative indexing notebooks (versions 1.0–1.4). It is shared as part of the article's commitment to full methodological transparency, enabling peer reviewers and other researchers to inspect, critique, and reproduce the corpus construction process.

**Note on this documentation:** The expanded commentary in this public version — including this header, inline explanatory notes, and the embedding model discrepancy note in Cell 9 — was drafted with the assistance of Claude (Anthropic), specifically Claude Sonnet 4.6, as part of the article preparation process. Code and methodology are the author's own.

---

## What This Notebook Produces

- `lincoln_speech_corpus.json` — runtime text store (~882 chunks); one JSON object per chunk with full text, metadata, and provenance fields
- `lincoln_index_embedded.csv` — embedding search index; same schema plus the `combined` field and serialized embedding vectors

---

## Design Decisions (v1.4)

- **Source for Miller Center texts:** `lincoln_speech_corpus.json` (original corpus JSON), not the Miller Center zip. Audit revealed zip file #58 (Second Annual Message, part 2) was truncated by 412 words, and source file #56 (Conkling letter) carried a wrong year label. Reading from the JSON eliminates both issues.
- **Chunking strategy (Approach C):** Paragraph-level targeting ~700 chars, capped at 900 chars, with sentence-split fallback for oversized paragraphs.
- **`MIN_CHUNK_CHARS` removed:** All fragments absorbed — zero content loss. The original minimum of 100 chars silently dropped short passages including the "Both read the same Bible" sentence in the Second Inaugural.
- **Lincoln-Douglas Debates:** Loaded by explicit chronological filename list. Alphabetical sort order caused systematic debate mislabeling in v1.2; explicit paths fix this.
- **Audience interjection markers preserved:** `[Cheers.]`, `[Laughter.]`, etc. are retained in debate texts as historically meaningful source artifacts. See Cell 5 for full rationale.
- **Post-chunking validation gate (Cell 6b):** Word counts compared against the reference JSON before any API calls. The notebook will not proceed to metadata generation unless all checks pass.

---

## Run Order

1. **Cells 1–6:** Parse, chunk, validate (no API calls, ~30 seconds)
2. **Cell 6b:** Validation gate — confirm all checks pass before proceeding
3. **Cell 7:** Generate metadata via GPT-4o-mini (~882 calls, ~8–10 min, resumable)
4. **Cells 8–9:** Generate embeddings (~9 batches, ~2–3 min, resumable)
5. **Cells 10–12:** Export and validate outputs

---

## Files Needed in `/content/`

- `lincoln_speech_corpus.json` — original corpus JSON (Miller Center texts + Emancipation Proclamation)
- `voyant_word_counts_complete.json` — Voyant Tools corpus frequency data (see Cell 2 for structure); generated by running the full Lincoln corpus through [Voyant Tools](https://voyant-tools.org) and exporting term frequency JSON
- Seven Lincoln-Douglas Debate `.txt` files (Lincoln-only, pre-filtered); exact filenames listed in Cell 1

---

## API Requirements and Estimated Costs

- **Cell 7 (metadata):** ~882 calls to `gpt-4o-mini`. Estimated cost at current pricing: ~$0.05–0.10 USD.
- **Cell 9 (embeddings):** ~9 batches of 100 to `text-embedding-3-small`. Estimated cost: <$0.01 USD.
- Set `OPENAI_API_KEY` as an environment variable (e.g., via Colab Secrets) before running. Do not hardcode keys in the notebook.

---

## Technical Notes

- **`\n\n` in `full_text`:** Paragraph breaks stored as literal `\n\n`. Ada-002 treats them as mild semantic boundaries (no adverse effect). Streamlit `st.markdown()` renders them as paragraph breaks. Hay keyword matching is newline-blind. CSV export round-trips them correctly via pandas quoting.
- **Resumability:** Both the metadata (Cell 7) and embedding (Cell 9) generation cells checkpoint progress to JSON files every N chunks. Safe to interrupt and resume without reprocessing completed chunks.


In [13]:
# Cell 1: Imports and Configuration

import os
import re
import json
import time
import pandas as pd
from pathlib import Path
from openai import OpenAI

# --- CONFIGURATION ---

# Original corpus JSON — used as the authoritative source for Miller Center texts.
# This bypasses the Miller Center zip entirely, eliminating two known zip issues:
#   (a) source_file #58 truncated by 412 words in the zip
#   (b) source_file #56 (Conkling) carried wrong year label in the zip
ORIGINAL_CORPUS_JSON = '/content/lincoln_speech_corpus.json'

VOYANT_JSON = '/content/voyant_word_counts_complete.json'

# Lincoln-Douglas Debate files — explicit chronological list.
# DO NOT replace with sorted(os.listdir(...)) or sorted(zf.namelist()):
# alphabetical filename order [Fifth, First, Fourth, Second, Seventh, Sixth, Third]
# does NOT match chronological order and caused systematic mislabeling in v1.2.
DEBATE_FILES = [
    '/content/First_Debate_-_Ottawa__Illinois__August_21__1858.txt',
    '/content/Second_Debate_-_Freepoint__Illinois__August_27__1858.txt',
    '/content/Third_Debate_-_Jonesboro__Illinois__September_15__1858.txt',
    '/content/Fourth_Debate_-_Charleston__Illinois__September_18__1858.txt',
    '/content/Fifth_Debate_-_Galesburg__Illinois__October_7__1858.txt',
    '/content/Sixth_Debate_-_Quincy__Illinois__October_13__1858.txt',
    '/content/Seventh_Debate_-_Alton___Illinois__October_15__1858.txt',
]

OUTPUT_JSON       = '/content/lincoln_speech_corpus_new.json'
OUTPUT_CSV        = '/content/lincoln_index_embedded.csv'
PROGRESS_FILE     = '/content/reindex_progress.json'
EMBEDDING_PROGRESS_FILE = '/content/embedding_progress.json'

# Chunking parameters
TARGET_CHUNK_CHARS       = 700
MAX_CHUNK_CHARS          = 900
SENTENCE_SPLIT_THRESHOLD = 900   # Matches MAX_CHUNK_CHARS — prevents oversized
                                  # accumulated chunks from multi-para merging.
                                  # Lowered from 1500 in v1.3.
# MIN_CHUNK_CHARS intentionally absent (removed in v1.3):
# The original MIN_CHUNK_CHARS = 100 silently dropped short fragments
# (rhetorical questions, single-sentence paragraphs), causing ~4,500 words of
# content loss including the 'Both read the same Bible' sentence in the
# Second Inaugural. All fragments are now absorbed regardless of length.

# API throttling
METADATA_DELAY_SECONDS   = 0.5
EMBEDDING_DELAY_SECONDS  = 1.0
EMBEDDING_BATCH_SIZE     = 100
METADATA_SAVE_EVERY      = 25

# Dry run mode — set False to make real API calls
DRY_RUN = False

# OpenAI client
client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

print('Configuration loaded.')
print(f'DRY_RUN = {DRY_RUN}')
print(f'Target chunk: {TARGET_CHUNK_CHARS} chars (max {MAX_CHUNK_CHARS}, split threshold {SENTENCE_SPLIT_THRESHOLD})')
print(f'MC source: {ORIGINAL_CORPUS_JSON}')
print(f'Debate files: {len(DEBATE_FILES)} listed explicitly in chronological order')


Configuration loaded.
DRY_RUN = False
Target chunk: 700 chars (max 900, split threshold 900)
MC source: /content/lincoln_speech_corpus.json
Debate files: 7 listed explicitly in chronological order


In [2]:
# Cell 2: Load Voyant Frequency Data
# Builds two lookup structures used by the metadata prompt in Cell 7:
#   voyant_freq: word -> rawFreq (integer)
#   rare_terms:  set of words with rawFreq 5-75 and inDocumentsCount >= 2
#                (the 'Hay-useful' sweet spot: distinctive but corpus-verified)

with open(VOYANT_JSON, 'r', encoding='utf-8') as f:
    voyant_data = json.load(f)

terms_list = voyant_data['corpusTerms']['terms']

# Full frequency lookup
voyant_freq = {t['term'].lower(): t['rawFreq'] for t in terms_list}

# Noise filters
stopwords = {
    'the','and','of','to','a','in','that','is','it','as','be','for',
    'with','by','not','on','or','from','at','this','his','their','he',
    'was','have','are','but','were','they','which','so','we','an','has',
    'been','had','will','would','all','its','our','more','upon','can',
    'if','may','shall','any','who','what','there','said','do','no',
    'these','those','them','my','me','us','him','her','she','i','you',
    'your','one','two','three','four','five','six','seven','eight','nine','ten',
    'mr','let','make','just','far','old','good','way','come','day','long',
    'know','set','read','past','high','meet','get','put','take','see',
    'give','go','think','fact','case','line','present','general','subject',
    'bill','act','world','must','much','also','such','other','very','well',
    'great','same','own','than','then','now','some','like','into','time',
    'upon','over','out','up','down','before','after','between','under','through'
}

def is_content_word(term):
    """Return True if term is a useful content word (not noise)."""
    if term.lower() in stopwords:
        return False
    if re.match(r'^[\d,\.\-\/%]+$', term):  # Pure numbers/punctuation
        return False
    if len(term) <= 3:
        return False
    return True

# Sweet-spot rare terms: freq 5-75, appears in 2+ documents, content word
rare_terms = set(
    t['term'].lower()
    for t in terms_list
    if 5 <= t['rawFreq'] <= 75
    and t['inDocumentsCount'] >= 2
    and is_content_word(t['term'])
)

# Very rare single-doc terms (freq 1-4) — also useful as discriminators
very_rare_terms = set(
    t['term'].lower()
    for t in terms_list
    if 1 <= t['rawFreq'] <= 4
    and is_content_word(t['term'])
)

print(f'Voyant corpus: {voyant_data["corpusTerms"]["total"]} total unique terms')
print(f'Sweet-spot rare terms (freq 5-75, 2+ docs): {len(rare_terms)}')
print(f'Very rare terms (freq 1-4): {len(very_rare_terms)}')
print()
print('Sample rare terms (historically significant):')
sample_hist = [t for t in sorted(rare_terms)
               if t in {'territories','emancipation','douglas','rebellion','proclamation',
                        'negro','freedom','popular','republican','fugitive','colored',
                        'colonization','secession','bondage','abolition','liberia',
                        'dred','scott','kansas','amendment','inaugur'}]
print(', '.join(sample_hist))

Voyant corpus: 7563 total unique terms
Sweet-spot rare terms (freq 5-75, 2+ docs): 2207
Very rare terms (freq 1-4): 4804

Sample rare terms (historically significant):
abolition, amendment, bondage, colonization, colored, dred, emancipation, freedom, fugitive, kansas, liberia, negro, popular, proclamation, rebellion, republican, scott, secession


In [3]:
# Cell 3: Sentence Splitter and Chunker (regex-based, no NLTK/spacy dependency)

def split_into_sentences(text):
    """
    Split text into sentences using regex.
    Handles common abbreviations and avoids false splits on Mr., Dr., etc.
    """
    # Protect common abbreviations from being split
    text = re.sub(r'\b(Mr|Mrs|Dr|Prof|Gov|Gen|Col|Lt|Sgt|Hon|Rev|Jr|Sr|St|Vol|No|vs|etc|viz|cf|approx)\.',
                  r'\1<PROTECT>', text)
    # Split on sentence-ending punctuation followed by space + capital
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)
    # Restore protected abbreviations
    sentences = [s.replace('<PROTECT>', '.') for s in sentences]
    return [s.strip() for s in sentences if s.strip()]


def chunk_by_paragraphs(paragraphs, target=TARGET_CHUNK_CHARS, max_chars=MAX_CHUNK_CHARS,
                         sentence_threshold=SENTENCE_SPLIT_THRESHOLD):
    """
    Group paragraphs into chunks targeting ~700 chars, capped at 900.

    Paragraphs over sentence_threshold are sentence-split first.
    ALL fragments are absorbed — there is no minimum length discard.
    Short fragments (rhetorical questions, single-sentence paragraphs) are
    appended to the current accumulator and will be sealed into a chunk
    once the accumulator reaches target length, or flushed at end-of-document.
    This ensures zero content loss regardless of source formatting.
    """
    # Pre-process: split oversized paragraphs into sentences
    processed = []
    for p in paragraphs:
        if len(p) > sentence_threshold:
            sents = split_into_sentences(p)
            processed.extend(sents)
        else:
            processed.append(p)

    chunks = []
    current_parts = []
    current_len = 0

    for part in processed:
        part = part.strip()
        if not part:          # skip genuinely empty strings only
            continue
        part_len = len(part)

        if current_len == 0:
            # Start a new chunk with this fragment regardless of its length
            current_parts.append(part)
            current_len = part_len
        elif current_len + part_len + 2 <= max_chars:
            # Fits within the hard cap — keep accumulating
            current_parts.append(part)
            current_len += part_len + 2
        elif current_len >= target:
            # Accumulator is at/above target — seal it, start fresh
            chunks.append('\n\n'.join(current_parts))
            current_parts = [part]
            current_len = part_len
        else:
            # Accumulator is below target — keep adding even if it exceeds max_chars
            # (a slightly oversized chunk is better than an orphaned fragment)
            current_parts.append(part)
            current_len += part_len + 2
            if current_len >= target:
                chunks.append('\n\n'.join(current_parts))
                current_parts = []
                current_len = 0

    # Flush any remaining parts
    if current_parts:
        chunks.append('\n\n'.join(current_parts))

    # Final guard: discard only genuinely empty strings (should never trigger)
    return [c for c in chunks if c.strip()]


print('Chunking functions defined (v1.3 — no MIN_CHUNK_CHARS drop).')


Chunking functions defined (v1.3 — no MIN_CHUNK_CHARS drop).


In [4]:
# Cell 4: Load Miller Center Texts from Original Corpus JSON
#
# v1.4 change: reads from lincoln_speech_corpus.json instead of the Miller Center zip.
#
# Why: audit of the pre-metadata corpus revealed two zip data quality issues:
#   (a) source_file #58 (Second Annual Message, part 2): zip file truncated after
#       paragraph 8 of 15, losing 412 words (financial/banking policy section).
#   (b) source_file #56 (Conkling letter): zip file carried 'August 26, 1864' date
#       label; correct date is August 26, 1863.
# The original corpus JSON is authoritative and free of both issues.
#
# Source label fixes still applied:
#   - Text #23 skipped (exact duplicate of #22, Henry Clay Eulogy)
#   - Annual Message date corrections applied for texts #59–64, #72–77
#     (zip files had wrong years; JSON also carries these corrections)

SKIP_TEXT_IDS = {23}

ANNUAL_MESSAGE_DATE_FIXES = {
    59: 'Source:  Second Annual Message. December 1, 1862',
    60: 'Source:  Second Annual Message. December 1, 1862',
    61: 'Source:  Second Annual Message. December 1, 1862',
    62: 'Source:  Second Annual Message. December 1, 1862',
    63: 'Source:  Second Annual Message. December 1, 1862',
    64: 'Source:  Second Annual Message. December 1, 1862',
    72: 'Source:  Fourth Annual Message. December 6, 1864',
    73: 'Source:  Fourth Annual Message. December 6, 1864',
    74: 'Source:  Fourth Annual Message. December 6, 1864',
    75: 'Source:  Fourth Annual Message. December 6, 1864',
    76: 'Source:  Fourth Annual Message. December 6, 1864',
    77: 'Source:  Fourth Annual Message. December 6, 1864',
}

def get_text_number(text_id_str):
    m = re.search(r'\d+', text_id_str)
    return int(m.group()) if m else -1


with open(ORIGINAL_CORPUS_JSON, 'r', encoding='utf-8') as f:
    original_corpus = json.load(f)

mc_raw_texts = []
skipped = []
date_fixes_applied = []

for entry in original_corpus:
    text_num = get_text_number(entry.get('text_id', ''))

    if text_num in SKIP_TEXT_IDS:
        skipped.append(f'#{text_num} (duplicate, skipped)')
        continue

    source = entry.get('source', '')
    # Normalise paragraph breaks: handle both literal \n\n and real newlines
    full_text = entry.get('full_text', '').replace('\\n\\n', '\n\n').strip()

    if not full_text:
        print(f'  WARNING: Text #{text_num} ({source}) has empty full_text — skipping.')
        continue

    # Apply date fixes
    if text_num in ANNUAL_MESSAGE_DATE_FIXES:
        old_source = source
        source = ANNUAL_MESSAGE_DATE_FIXES[text_num]
        date_fixes_applied.append(f'  #{text_num}: "{old_source}" → "{source}"')

    mc_raw_texts.append({
        'text_num':   text_num,
        'source':     source,
        'full_text':  full_text,
        'source_type': 'miller_center',
    })

mc_raw_texts.sort(key=lambda x: x['text_num'])

print(f'Loaded {len(mc_raw_texts)} Miller Center texts from JSON ({len(skipped)} skipped)')
if skipped:
    for s in skipped: print(f'  Skipped: {s}')
if date_fixes_applied:
    print(f'Date fixes applied ({len(date_fixes_applied)}):')
    for fix in date_fixes_applied: print(fix)

# Sanity check: word count vs expected baseline
total_mc_words = sum(len(t['full_text'].split()) for t in mc_raw_texts)
EXPECTED_MC_WORDS = 73373  # baseline from audit of original JSON excl. duplicate #23
diff = abs(total_mc_words - EXPECTED_MC_WORDS)
status = '✅' if diff <= 20 else '⚠️  UNEXPECTED DIFFERENCE'
print(f'\nMC word count: {total_mc_words} (expected ~{EXPECTED_MC_WORDS}) {status}')
if diff > 20:
    print(f'  Diff = {total_mc_words - EXPECTED_MC_WORDS:+d} words. '
          f'Check that the correct lincoln_speech_corpus.json was uploaded.')


Loaded 79 Miller Center texts from JSON (1 skipped)
  Skipped: #23 (duplicate, skipped)
Date fixes applied (12):
  #59: "Source:  Second Annual Message. December 1, 1862" → "Source:  Second Annual Message. December 1, 1862"
  #60: "Source:  Second Annual Message. December 1, 1862" → "Source:  Second Annual Message. December 1, 1862"
  #61: "Source:  Second Annual Message. December 1, 1862" → "Source:  Second Annual Message. December 1, 1862"
  #62: "Source:  Second Annual Message. December 1, 1862" → "Source:  Second Annual Message. December 1, 1862"
  #63: "Source:  Second Annual Message. December 1, 1862" → "Source:  Second Annual Message. December 1, 1862"
  #64: "Source:  Second Annual Message. December 1, 1862" → "Source:  Second Annual Message. December 1, 1862"
  #72: "Source:  Fourth Annual Message. December 6, 1864." → "Source:  Fourth Annual Message. December 6, 1864"
  #73: "Source:  Fourth Annual Message. December 6, 1864." → "Source:  Fourth Annual Message. December 6, 186

In [5]:
# Cell 4b: Emancipation Proclamation — handled automatically via JSON
#
# In v1.3 and earlier, this cell required manually pasting the Emancipation
# Proclamation text because the Miller Center zip file (Text #54) had an empty
# full_text field. In v1.4, Cell 4 reads from the original corpus JSON, which
# already contains the Emancipation Proclamation text correctly.
#
# This cell is kept as a placeholder to preserve cell numbering.
# No action required — proceed to Cell 5.

ep_chunks = [t for t in mc_raw_texts if 'Emancipation' in t.get('source', '')]
if ep_chunks:
    ep_words = sum(len(t['full_text'].split()) for t in ep_chunks)
    print(f'✅ Emancipation Proclamation: {len(ep_chunks)} source entries, {ep_words} words — loaded from JSON.')
else:
    print('⚠️  Emancipation Proclamation not found. Check that lincoln_speech_corpus.json is correct.')


✅ Emancipation Proclamation: 1 source entries, 724 words — loaded from JSON.


In [7]:
# Cell 5: Parse Lincoln-Douglas Debates .txt Files
#
# File format:
#   Plain text with actual \n\n paragraph breaks (not literal \\n\\n)
#   Files are Lincoln-only (pre-filtered)
#
# IMPORTANT (v1.3): Files are loaded from DEBATE_FILES (defined in Cell 1),
# an explicit chronological list. The previous version used sorted(zf.namelist())
# on a zip archive, which produced alphabetical filename order:
#   [Fifth, First, Fourth, Second, Seventh, Sixth, Third]
# This did NOT match DEBATE_ORDER (chronological), so every debate except the
# Sixth/Quincy was assigned the wrong content and wrong source label.
# The fix: iterate DEBATE_FILES and DEBATE_ORDER together by index.
#
# METHODOLOGICAL DECISION — Interjections preserved intentionally:
#   Audience interjection markers ([Cheers.], [Laughter.], [Great applause.], etc.)
#   are retained rather than stripped. These markers are part of the original newspaper
#   transcriptions and represent the performative, dialogic dimension of Lincoln's
#   rhetoric absent from the Miller Center texts. Preserving them:
#     (a) maintains fidelity to the source as published
#     (b) allows the system to surface chunks relevant to questions about humor,
#         crowd response, and Lincoln's improvisational register
#     (c) provides an honest test of RAG performance on artifact-laden historical corpora
#   Expected side effects: minor keyword signal dilution; occasional weak metadata
#   summaries for interjection-heavy chunks. Acceptable.

DEBATE_ORDER = [
    ('First Debate',   'Ottawa, Illinois, August 21, 1858'),
    ('Second Debate',  'Freeport, Illinois, August 27, 1858'),
    ('Third Debate',   'Jonesboro, Illinois, September 15, 1858'),
    ('Fourth Debate',  'Charleston, Illinois, September 18, 1858'),
    ('Fifth Debate',   'Galesburg, Illinois, October 7, 1858'),
    ('Sixth Debate',   'Quincy, Illinois, October 13, 1858'),
    ('Seventh Debate', 'Alton, Illinois, October 15, 1858'),
]

def clean_debate_text(text):
    """Clean debate text while preserving audience interjection markers.

    Changes from v1.2:
    - Normalises \r\n to \n before any other processing, eliminating
      mixed line-ending inconsistencies between debate files.
    """
    # Normalise Windows line endings first
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    # Remove speaker headers like "Mr. Lincoln's Speech" and "Mr. Lincoln's Reply"
    text = re.sub(r"Mr\.?\s+Lincoln'?s?\s+(Speech|Reply)\.?", '', text, flags=re.IGNORECASE)
    # Normalize whitespace (multiple spaces and excess blank lines only)
    text = re.sub(r' {2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


ld_raw_texts = []

assert len(DEBATE_FILES) == len(DEBATE_ORDER), (
    f'DEBATE_FILES ({len(DEBATE_FILES)}) and DEBATE_ORDER ({len(DEBATE_ORDER)}) '
    f'must have the same length'
)

for i, (fname, (debate_name, debate_location_date)) in enumerate(
        zip(DEBATE_FILES, DEBATE_ORDER)):

    fpath = Path(fname)
    if not fpath.exists():
        raise FileNotFoundError(
            f'Debate file not found: {fpath}\n'
            f'Check DEBATES_DIR = {repr(DEBATES_DIR)} in Cell 1.'
        )

    raw = fpath.read_text(encoding='utf-8', errors='replace')
    source = f'Lincoln-Douglas Debates, {debate_name}, {debate_location_date}'
    interjection_count = len(re.findall(r'\[.*?\]', raw))
    cleaned = clean_debate_text(raw)

    print(f'  {fname}:')
    print(f'    → assigned label : {debate_name}, {debate_location_date}')
    print(f'    → {len(cleaned):,} chars, {interjection_count} interjections preserved')

    ld_raw_texts.append({
        'debate_index': i,
        'debate_name': debate_name,
        'source': source,
        'full_text': cleaned,
        'source_type': 'lincoln_douglas',
    })

print(f'\nLoaded {len(ld_raw_texts)} Lincoln-Douglas Debate files')
print('Audience interjection markers preserved in all debate texts (methodological decision).')

  /content/First_Debate_-_Ottawa__Illinois__August_21__1858.txt:
    → assigned label : First Debate, Ottawa, Illinois, August 21, 1858
    → 47,748 chars, 58 interjections preserved
  /content/Second_Debate_-_Freepoint__Illinois__August_27__1858.txt:
    → assigned label : Second Debate, Freeport, Illinois, August 27, 1858
    → 44,833 chars, 59 interjections preserved
  /content/Third_Debate_-_Jonesboro__Illinois__September_15__1858.txt:
    → assigned label : Third Debate, Jonesboro, Illinois, September 15, 1858
    → 51,861 chars, 17 interjections preserved
  /content/Fourth_Debate_-_Charleston__Illinois__September_18__1858.txt:
    → assigned label : Fourth Debate, Charleston, Illinois, September 18, 1858
    → 56,824 chars, 81 interjections preserved
  /content/Fifth_Debate_-_Galesburg__Illinois__October_7__1858.txt:
    → assigned label : Fifth Debate, Galesburg, Illinois, October 7, 1858
    → 45,526 chars, 5 interjections preserved
  /content/Sixth_Debate_-_Quincy__Illinois__O

In [8]:
# Cell 6: Chunk All Texts and Build Chunk List
#
# Assigns sequential text_id (Text #: 0, 1, 2, ...) across all chunks.
# Adds position and total_in_source for the adjacency index.
# Adds chunk_context (deterministic: 'From [speech], passage N of M').

all_chunks = []  # Final list of chunk dicts

def paragraphs_from_miller_center(full_text):
    """Split Miller Center text on literal \\n\\n markers."""
    return [p.strip() for p in full_text.split('\n\n') if p.strip()]

def paragraphs_from_debates(full_text):
    """Split debate text on actual newlines."""
    return [p.strip() for p in full_text.split('\n\n') if p.strip()]


chunk_id = 0

# --- Miller Center texts ---
mc_chunk_count = 0
for text in mc_raw_texts:
    if not text['full_text'] or text['full_text'].startswith('PASTE'):
        print(f"  Skipping Text #{text['text_num']} — no full_text")
        continue

    paragraphs = paragraphs_from_miller_center(text['full_text'])
    chunks = chunk_by_paragraphs(paragraphs)
    total = len(chunks)

    for pos, chunk_text in enumerate(chunks):
        # Strip 'Source:  ' prefix from Miller Center source field for clean chunk_context
        clean_source = re.sub(r'^Source:\s+', '', text['source']).strip()
        chunk_context = f"From Lincoln's {clean_source}, passage {pos+1} of {total}."

        all_chunks.append({
            'text_id': f'Text #: {chunk_id}',
            'source': text['source'],
            'full_text': chunk_text,
            'summary': '',           # To be generated in Cell 7
            'keywords': '',          # To be generated in Cell 7
            'keywords_semantic': '', # To be generated in Cell 7
            'chunk_context': chunk_context,
            'position': pos,
            'total_in_source': total,
            'source_file': text['text_num'],
            'source_type': 'miller_center',
            'combined': '',          # Built after metadata generation
            'embedding': [],         # Generated in Cells 8-9
        })
        chunk_id += 1

    mc_chunk_count += total

# --- Lincoln-Douglas Debate texts ---
ld_chunk_count = 0
for text in ld_raw_texts:
    paragraphs = paragraphs_from_debates(text['full_text'])
    chunks = chunk_by_paragraphs(paragraphs)
    total = len(chunks)

    for pos, chunk_text in enumerate(chunks):
        chunk_context = f"From the {text['source']}, passage {pos+1} of {total}."

        all_chunks.append({
            'text_id': f'Text #: {chunk_id}',
            'source': text['source'],
            'full_text': chunk_text,
            'summary': '',
            'keywords': '',
            'keywords_semantic': '',
            'chunk_context': chunk_context,
            'position': pos,
            'total_in_source': total,
            'source_file': f'debate_{text["debate_index"]}',
            'source_type': 'lincoln_douglas',
            'combined': '',
            'embedding': [],
        })
        chunk_id += 1

    ld_chunk_count += total


# --- Size distribution report ---
sizes = [len(c['full_text']) for c in all_chunks]
buckets = [(0,200),(200,400),(400,600),(600,800),(800,1000),(1000,1500),(1500,9999)]

print(f'\n=== CHUNK SUMMARY ===')
print(f'Total chunks: {len(all_chunks)}')
print(f'  Miller Center: {mc_chunk_count}')
print(f'  Lincoln-Douglas Debates: {ld_chunk_count}')
print(f'Mean chunk size: {sum(sizes)/len(sizes):.0f} chars')
print(f'Min: {min(sizes)} | Max: {max(sizes)}')
print()
print('Size distribution:')
for lo, hi in buckets:
    count = sum(1 for s in sizes if lo <= s < hi)
    pct = 100 * count / len(sizes)
    label = f'{lo}-{hi-1}' if hi < 9999 else f'{lo}+'
    print(f'  {label:>10} chars: {count:4d} chunks ({pct:.1f}%)')
if max(sizes) >= SENTENCE_SPLIT_THRESHOLD:
    print(f'\nWARNING: {sum(1 for s in sizes if s >= SENTENCE_SPLIT_THRESHOLD)} chunks exceed {SENTENCE_SPLIT_THRESHOLD} chars — check sentence splitter')


=== CHUNK SUMMARY ===
Total chunks: 886
  Miller Center: 487
  Lincoln-Douglas Debates: 399
Mean chunk size: 889 chars
Min: 68 | Max: 1863

Size distribution:
       0-199 chars:    6 chunks (0.7%)
     200-399 chars:   15 chunks (1.7%)
     400-599 chars:   18 chunks (2.0%)
     600-799 chars:  204 chunks (23.0%)
     800-999 chars:  457 chunks (51.6%)
   1000-1499 chars:  181 chunks (20.4%)
       1500+ chars:    5 chunks (0.6%)



In [9]:
# Cell 6b: Post-Chunking Validation
#
# Compares word counts in all_chunks against the reference JSON before any API calls.
# Run this cell and confirm ALL checks pass before proceeding to Cell 7 (metadata).
# If any check fails, do NOT proceed — diagnose and re-run Cells 4–6 first.

from collections import defaultdict

SKIP_TEXT_IDS_VAL = {23}
ANNUAL_FIXES_VAL = {
    59:'Source:  Second Annual Message. December 1, 1862',
    60:'Source:  Second Annual Message. December 1, 1862',
    61:'Source:  Second Annual Message. December 1, 1862',
    62:'Source:  Second Annual Message. December 1, 1862',
    63:'Source:  Second Annual Message. December 1, 1862',
    64:'Source:  Second Annual Message. December 1, 1862',
    72:'Source:  Fourth Annual Message. December 6, 1864',
    73:'Source:  Fourth Annual Message. December 6, 1864',
    74:'Source:  Fourth Annual Message. December 6, 1864',
    75:'Source:  Fourth Annual Message. December 6, 1864',
    76:'Source:  Fourth Annual Message. December 6, 1864',
    77:'Source:  Fourth Annual Message. December 6, 1864',
}

# ── 1. Build expected word counts from reference JSON ──────────────────────
expected_mc = defaultdict(int)
for entry in original_corpus:
    n = get_text_number(entry.get('text_id', ''))
    if n in SKIP_TEXT_IDS_VAL:
        continue
    src = ANNUAL_FIXES_VAL.get(n, entry.get('source', ''))
    ft = entry.get('full_text', '').replace('\\n\\n', '\n\n').strip()
    expected_mc[src] += len(ft.split())

# ── 2. Build actual word counts from all_chunks ─────────────────────────────
actual_mc = defaultdict(int)
actual_mc_chunks = defaultdict(int)
actual_ld = defaultdict(int)
actual_ld_chunks = defaultdict(int)

for chunk in all_chunks:
    w = len(chunk['full_text'].split())
    if chunk['source_type'] == 'miller_center':
        actual_mc[chunk['source']] += w
        actual_mc_chunks[chunk['source']] += 1
    else:
        actual_ld[chunk['source']] += w
        actual_ld_chunks[chunk['source']] += 1

# ── 3. Miller Center comparison ─────────────────────────────────────────────
print('=== MILLER CENTER WORD COUNT VALIDATION ===')
mc_failures = []
for src in sorted(expected_mc):
    exp = expected_mc[src]
    act = actual_mc.get(src, 0)
    diff = exp - act
    ok = abs(diff) <= 5   # allow ≤5 words for tokenization variation
    flag = '✅' if ok else f'❌ DIFF={diff:+d}'
    if not ok:
        mc_failures.append((src, exp, act, diff))
    chunks_n = actual_mc_chunks.get(src, 0)
    print(f'  {flag}  {chunks_n:3d} chunks  exp={exp:5d} act={act:5d}  {src[:60]}')

# ── 4. Lincoln-Douglas comparison ──────────────────────────────────────────
LD_EXPECTED = {
    'Lincoln-Douglas Debates, First Debate, Ottawa, Illinois, August 21, 1858':      8778,
    'Lincoln-Douglas Debates, Second Debate, Freeport, Illinois, August 27, 1858':   8089,
    'Lincoln-Douglas Debates, Third Debate, Jonesboro, Illinois, September 15, 1858': 9365,
    'Lincoln-Douglas Debates, Fourth Debate, Charleston, Illinois, September 18, 1858': 10485,
    'Lincoln-Douglas Debates, Fifth Debate, Galesburg, Illinois, October 7, 1858':   8154,
    'Lincoln-Douglas Debates, Sixth Debate, Quincy, Illinois, October 13, 1858':     9417,
    'Lincoln-Douglas Debates, Seventh Debate, Alton, Illinois, October 15, 1858':   10364,
}

print()
print('=== LINCOLN-DOUGLAS WORD COUNT VALIDATION ===')
ld_failures = []
for src, exp in sorted(LD_EXPECTED.items()):
    act = actual_ld.get(src, 0)
    diff = exp - act
    ok = diff == 0
    flag = '✅' if ok else f'❌ DIFF={diff:+d}'
    if not ok:
        ld_failures.append((src, exp, act, diff))
    short = src.split(', ')[1]
    print(f'  {flag}  {actual_ld_chunks.get(src,0):3d} chunks  exp={exp:5d} act={act:5d}  {short}')

# ── 5. Structural checks ────────────────────────────────────────────────────
print()
print('=== STRUCTURAL CHECKS ===')

# Sequential IDs
ids = [int(c['text_id'].split('#: ')[1]) for c in all_chunks]
seq_ok = ids == list(range(len(all_chunks)))
print(f'  Sequential IDs 0–{len(all_chunks)-1}: {"✅" if seq_ok else "❌ BROKEN"}')

# No empty full_text
empty = [c['text_id'] for c in all_chunks if not c['full_text'].strip()]
print(f'  Empty full_text: {"✅ none" if not empty else f"❌ {len(empty)} found: {empty[:5]}"}')

# Source type counts
from collections import Counter
src_types = Counter(c['source_type'] for c in all_chunks)
print(f'  Source types: {dict(src_types)}')
print(f'  Total chunks: {len(all_chunks)}')

# Chunk size distribution
sizes = [len(c['full_text']) for c in all_chunks]
over_1500 = sum(1 for s in sizes if s > 1500)
print(f'  Mean chunk size: {sum(sizes)/len(sizes):.0f} chars')
print(f'  Min: {min(sizes)}  Max: {max(sizes)}')
print(f'  Over 1500 chars: {over_1500} (expected ≤10 irreducible long sentences)')

# Key passage spot checks
print()
print('=== KEY PASSAGE SPOT CHECKS ===')
spot_checks = [
    ('Second Inaugural — Bible passage',
     'Both read the same Bible'),
    ('Second Inaugural — Prayers of both',
     'The prayers of both could not be answered'),
    ('Second Inaugural — Fondly do we hope',
     'Fondly do we hope'),
    ('Conkling — correct year label',
     'August 26, 1863'),
    ('Second Annual — specie payments (previously truncated)',
     'suspension of specie payments'),
    ('Second Annual — banking associations (previously truncated)',
     'furnish circulating notes'),
    ('Ottawa/First Debate — house divided reference',
     'house divided'),
    ('Charleston/Fourth Debate — physical difference',
     'physical difference between the white and black races'),
]
spot_failures = []
for label, phrase in spot_checks:
    found = [c['text_id'] for c in all_chunks
             if phrase.lower() in c['full_text'].lower()]
    ok = len(found) > 0
    if not ok: spot_failures.append(label)
    print(f'  {"✅" if ok else "❌"}  {label}: {found if ok else "NOT FOUND"}')

# ── 6. Go / No-Go summary ──────────────────────────────────────────────────
print()
print('=== GO / NO-GO ===')
all_pass = (not mc_failures and not ld_failures and
            seq_ok and not empty and not spot_failures)
if all_pass:
    print('✅ ALL CHECKS PASSED — safe to proceed to Cell 7 (metadata generation)')
else:
    print('❌ FAILURES DETECTED — do NOT proceed to Cell 7')
    if mc_failures:
        print(f'  MC failures ({len(mc_failures)}):')
        for src, exp, act, diff in mc_failures:
            print(f'    {src[:60]}: exp={exp} act={act} diff={diff:+d}')
    if ld_failures:
        print(f'  LD failures ({len(ld_failures)}):')
        for src, exp, act, diff in ld_failures:
            print(f'    {src[:60]}: exp={exp} act={act} diff={diff:+d}')
    if spot_failures:
        print(f'  Missing passages: {spot_failures}')
    if not seq_ok: print('  Non-sequential chunk IDs')
    if empty: print(f'  Empty chunks: {empty}')


=== MILLER CENTER WORD COUNT VALIDATION ===
  ✅   11 chunks  exp= 1670 act= 1670  Source:   Public Letter to James Conkling. August 26, 1863
  ✅   20 chunks  exp= 3183 act= 3183  Source:  A House Divided Speech. June 16, 1858.
  ✅  107 chunks  exp=16259 act=16259  Source:  At Peoria, Illinois. October 16, 1854.
  ✅   51 chunks  exp= 7715 act= 7715  Source:  Cooper Union Address. February 27, 1860.
  ✅    5 chunks  exp=  724 act=  724  Source:  Emancipation Proclamation. January 1, 1863.
  ✅   37 chunks  exp= 5349 act= 5349  Source:  Eulogy on Henry Clay. July 6, 1852.
  ✅    1 chunks  exp=  151 act=  151  Source:  Farewell Address. February 11, 1861.
  ✅   44 chunks  exp= 7117 act= 7117  Source:  First Annual Message. December 3, 1861.
  ✅   22 chunks  exp= 3621 act= 3621  Source:  First Inaugural Address. March 4, 1861.
  ✅   41 chunks  exp= 5860 act= 5860  Source:  Fourth Annual Message. December 6, 1864
  ✅    1 chunks  exp=  264 act=  264  Source:  Gettysburg Address. November 19, 

In [14]:
# Cell 6c: Metadata Preview Run (first 5 chunks)
#
# Runs the metadata API on chunks 0–4 only, prints full output for inspection.
# Does NOT write to the progress file — safe to run without affecting Cell 7.
# Review output before committing to the full run.

preview_chunks = all_chunks[:5]

print(f'=== METADATA PREVIEW: chunks 0–4 ===')
print(f'Model: gpt-4o-mini  |  DRY_RUN = {DRY_RUN}')
print()

for chunk in preview_chunks:
    print(f'{"─" * 70}')
    print(f'text_id:       {chunk["text_id"]}')
    print(f'source:        {chunk["source"]}')
    print(f'chunk_context: {chunk["chunk_context"]}')
    print(f'full_text ({len(chunk["full_text"])} chars):')
    print(chunk['full_text'])
    print()

    meta = generate_metadata(chunk, rare_terms, very_rare_terms, dry_run=DRY_RUN)

    print(f'SUMMARY:')
    print(f'  {meta["summary"]}')
    print(f'KEYWORDS:')
    print(f'  {meta["keywords"]}')
    print(f'KEYWORDS_SEMANTIC:')
    print(f'  {meta["keywords_semantic"]}')
    print()

print(f'{"─" * 70}')
print(f'Preview complete. If output looks good, proceed to Cell 7.')
print(f'Set DRY_RUN = False in Cell 1 before running Cell 7.')

=== METADATA PREVIEW: chunks 0–4 ===
Model: gpt-4o-mini  |  DRY_RUN = False

──────────────────────────────────────────────────────────────────────
text_id:       Text #: 0
source:        Source:  At Peoria, Illinois. October 16, 1854.
chunk_context: From Lincoln's At Peoria, Illinois. October 16, 1854., passage 1 of 5.
full_text (883 chars):
"The repeal of the Missouri Compromise, and the propriety of its restoration, constitute the subject of what I am about to say.

As I desire to present my own connected view of this subject, my remarks will not be, specifically, an answer to Judge Douglas; yet, as I proceed, the main points he has presented will arise, and will receive such respectful attention as I may be able to give them.

I wish further to say, that I do not propose to question the patriotism, or to assail the motives of any man, or class of men; but rather to strictly confine myself to the naked merits of the question.

I also wish to be no less than National in all the posit

In [15]:
# Cell 7: Generate Metadata (Summary, Keywords, Keywords_Semantic) via GPT-4o-mini
#
# One API call per chunk combining all three metadata fields.
# Uses Voyant rare_terms as guidance for keyword selection.
# Progress saved every METADATA_SAVE_EVERY chunks — safe to interrupt and resume.
#
# IMPORTANT: Set DRY_RUN = False in Cell 1 before running this cell.

def build_rare_term_hint(chunk_text, rare_terms, very_rare_terms, top_n=20):
    """
    Find Voyant-verified rare terms that actually appear in this chunk.
    Returns a string of candidate terms for the metadata prompt.
    """
    chunk_lower = chunk_text.lower()
    # Words present in chunk
    chunk_words = set(re.findall(r'\b[a-z]{4,}\b', chunk_lower))

    # Sweet-spot rare terms present in chunk (prefer these)
    present_rare = [w for w in rare_terms if w in chunk_words]
    # Very rare terms present in chunk (strong discriminators)
    present_very_rare = [w for w in very_rare_terms if w in chunk_words]

    # Combine, deduplicate, limit
    candidates = list(dict.fromkeys(present_rare + present_very_rare))[:top_n]
    return ', '.join(candidates) if candidates else 'none identified'


def generate_metadata(chunk, rare_terms, very_rare_terms, dry_run=True):
    """
    Call GPT-4o-mini to generate summary, keywords, and keywords_semantic.
    Returns dict with those three fields, or placeholder strings if dry_run=True.
    """
    if dry_run:
        return {
            'summary': '[DRY RUN SUMMARY]',
            'keywords': ['dry', 'run', 'keywords'],  # List format matching production output
            'keywords_semantic': 'dry run concepts',
        }

    rare_hint = build_rare_term_hint(chunk['full_text'], rare_terms, very_rare_terms)

    prompt = f"""You are a historian generating metadata for a passage from Lincoln's speeches,
for use in a retrieval-augmented generation (RAG) system.

Context: {chunk['chunk_context']}

Passage:
{chunk['full_text']}

Generate three metadata fields:

1. SUMMARY: A brief 1-2 sentence description of the specific argument or content in this passage.
   Be specific — name the actual claim, policy, or rhetorical move Lincoln makes.
   Do NOT write generic summaries like "Lincoln discusses slavery."
   Frame as: "Lincoln argues that..." or "In this passage, Lincoln..."

2. KEYWORDS: A comma-separated list of 10-15 single words for retrieval search.
   CRITICAL RULES for keywords:
   - Prioritize RARE, SPECIFIC terms over common ones. Prefer words that appear rarely in Lincoln's corpus.
   - Corpus-rare term candidates present in this passage: {rare_hint}
   - Include proper nouns (names, places, legislation) when present
   - Include historically specific terms (e.g. "Nebraska", "secession", "bondage", "colonization")
   - AVOID generic words like: government, people, states, nation, law, war, time, great, men, words
   - Single words ONLY — no phrases

3. KEYWORDS_SEMANTIC: A comma-separated list of 5-8 short concept phrases for semantic search.
   These supplement the single-word keywords for embedding search.
   Include historiographically significant concepts even if multi-word:
   Examples: "popular sovereignty", "Dred Scott decision", "Black citizenship", "free labor ideology",
   "border state politics", "colonization scheme", "second inaugural address"
   Aim for concepts a historian would search for, not just topic labels.

Respond in this exact format:
SUMMARY: [your summary]
KEYWORDS: [word1, word2, word3, ...]
KEYWORDS_SEMANTIC: [phrase1, phrase2, ...]"""

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=400,
        temperature=0.3,
    )

    text = response.choices[0].message.content.strip()

    # Parse response
    summary = ''
    keywords = ''
    keywords_semantic = ''

    for line in text.split('\n'):
        if line.startswith('SUMMARY:'):
            summary = line[len('SUMMARY:'):].strip()
        elif line.startswith('KEYWORDS:'):
            keywords = line[len('KEYWORDS:'):].strip()
        elif line.startswith('KEYWORDS_SEMANTIC:'):
            keywords_semantic = line[len('KEYWORDS_SEMANTIC:'):].strip()

    # Convert comma-separated keyword string to list for app compatibility
    # The app does ' '.join(entry['keywords']) — requires list, not string
    keywords_list = [k.strip() for k in keywords.split(',') if k.strip()] if keywords else []

    return {
        'summary': summary,
        'keywords': keywords_list,
        'keywords_semantic': keywords_semantic,
    }


# --- Load existing progress if resuming ---
# NOTE: The progress file keys metadata by text_id string (e.g. 'Text #: 42').
# If the corpus has been re-chunked since the progress file was written, chunk
# IDs will have shifted and the old progress entries will simply not match any
# current chunk — the resume logic skips non-matching IDs silently, so all
# chunks will be treated as needing fresh generation. This is correct behaviour.
# Do NOT manually edit text_ids in the progress file to force reuse of old metadata
# — the metadata was written for different chunk content.
progress = {}
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE, 'r') as f:
        progress = json.load(f)
    print(f'Resuming from progress file: {len(progress)} chunks already processed')
    print('  (IDs not matching current chunks will be regenerated automatically)')
else:
    print('Starting fresh \u2014 no progress file found')


# --- Generate metadata ---
to_process = [c for c in all_chunks if c['text_id'] not in progress]
print(f'Chunks to process: {len(to_process)} (of {len(all_chunks)} total)')

if DRY_RUN:
    print('DRY RUN — no API calls will be made')

for i, chunk in enumerate(to_process):
    meta = generate_metadata(chunk, rare_terms, very_rare_terms, dry_run=DRY_RUN)

    chunk['summary'] = meta['summary']
    chunk['keywords'] = meta['keywords']
    chunk['keywords_semantic'] = meta['keywords_semantic']

    # Store in progress dict
    progress[chunk['text_id']] = meta

    # Save progress periodically
    if (i + 1) % METADATA_SAVE_EVERY == 0:
        with open(PROGRESS_FILE, 'w') as f:
            json.dump(progress, f)
        print(f'  Progress saved: {i+1}/{len(to_process)} processed')

    if not DRY_RUN:
        time.sleep(METADATA_DELAY_SECONDS)

# Apply any progress-restored metadata to chunks that were already done
for chunk in all_chunks:
    if chunk['text_id'] in progress and not chunk['summary']:
        meta = progress[chunk['text_id']]
        chunk['summary'] = meta['summary']
        chunk['keywords'] = meta['keywords']
        chunk['keywords_semantic'] = meta['keywords_semantic']

# Final progress save
with open(PROGRESS_FILE, 'w') as f:
    json.dump(progress, f)

print(f'\nMetadata generation complete. Total chunks: {len(all_chunks)}')

Starting fresh — no progress file found
Chunks to process: 886 (of 886 total)
  Progress saved: 25/886 processed
  Progress saved: 50/886 processed
  Progress saved: 75/886 processed
  Progress saved: 100/886 processed
  Progress saved: 125/886 processed
  Progress saved: 150/886 processed
  Progress saved: 175/886 processed
  Progress saved: 200/886 processed
  Progress saved: 225/886 processed
  Progress saved: 250/886 processed
  Progress saved: 275/886 processed
  Progress saved: 300/886 processed
  Progress saved: 325/886 processed
  Progress saved: 350/886 processed
  Progress saved: 375/886 processed
  Progress saved: 400/886 processed
  Progress saved: 425/886 processed
  Progress saved: 450/886 processed
  Progress saved: 475/886 processed
  Progress saved: 500/886 processed
  Progress saved: 525/886 processed
  Progress saved: 550/886 processed
  Progress saved: 575/886 processed
  Progress saved: 600/886 processed
  Progress saved: 625/886 processed
  Progress saved: 650/886

In [16]:
# Cell 8: Build 'combined' Field for Each Chunk
#
# The combined field is what gets embedded. It concatenates all semantic content:
# text_id, source, chunk_context, full_text, summary, keywords, keywords_semantic.
# This design co-embeds structural, content, and metadata signals.

def build_combined(chunk):
    parts = [
        chunk['text_id'],
        f"Source: {chunk['source']}",
        chunk['chunk_context'],
        chunk['full_text'],
    ]
    if chunk['summary']:
        parts.append(f"Summary: {chunk['summary']}")
    if chunk['keywords']:
        # keywords is stored as a list; join for the combined embedding string
        kw_str = ', '.join(chunk['keywords']) if isinstance(chunk['keywords'], list) else chunk['keywords']
        parts.append(f"Keywords: {kw_str}")
    if chunk['keywords_semantic']:
        parts.append(f"Concepts: {chunk['keywords_semantic']}")
    return '\n\n'.join(parts)

for chunk in all_chunks:
    chunk['combined'] = build_combined(chunk)

combined_lengths = [len(c['combined']) for c in all_chunks]
print(f'Combined field built for {len(all_chunks)} chunks')
print(f'Mean combined length: {sum(combined_lengths)/len(combined_lengths):.0f} chars')
print(f'Max combined length: {max(combined_lengths)} chars')
# Ada-002 limit is 8,191 tokens; ~4 chars/token → ~32,764 chars. Well within limits.
over_limit = [c['text_id'] for c in all_chunks if len(c['combined']) > 30000]
if over_limit:
    print(f'WARNING: {len(over_limit)} chunks have combined field > 30,000 chars: {over_limit}')
else:
    print('All chunks within Ada-002 token limits.')

Combined field built for 886 chunks
Mean combined length: 1640 chars
Max combined length: 2635 chars
All chunks within Ada-002 token limits.


In [17]:
# Cell 9: Generate Embeddings via text-embedding-3-small
#
# Batches of EMBEDDING_BATCH_SIZE combined strings per API call.
# Progress tracked by chunk index — safe to interrupt and resume.
# Embeddings stored as lists (JSON-serializable).
#
# NOTE ON EMBEDDING MODEL:
# This notebook uses text-embedding-3-small. The production Nicolay system
# uses text-embedding-ada-002 (loaded from a pre-built index file). The
# discrepancy between this notebook and the production index is intentional
# and documented: a controlled embedding model comparison (ada-002 vs.
# text-embedding-3-small) forms part of the article's evaluation findings.
# See the Retrieval Architecture and Benchmarking sections of the article
# for full discussion. Researchers replicating the production system should
# use text-embedding-ada-002.

# EMBEDDING_PROGRESS_FILE defined in Cell 1

# Load embedding progress if resuming
embedding_progress = {}  # text_id -> embedding list
if os.path.exists(EMBEDDING_PROGRESS_FILE):
    with open(EMBEDDING_PROGRESS_FILE, 'r') as f:
        embedding_progress = json.load(f)
    print(f'Resuming embeddings: {len(embedding_progress)} already done')

# Apply already-computed embeddings
for chunk in all_chunks:
    if chunk['text_id'] in embedding_progress:
        chunk['embedding'] = embedding_progress[chunk['text_id']]

# Chunks that still need embeddings
needs_embedding = [c for c in all_chunks if not c['embedding']]
print(f'Chunks needing embeddings: {len(needs_embedding)}')

if DRY_RUN:
    print('DRY RUN — skipping embedding generation')
    # Assign dummy embeddings for dry run
    for chunk in needs_embedding:
        chunk['embedding'] = [0.0] * 1536
else:
    # Process in batches
    batches = [needs_embedding[i:i+EMBEDDING_BATCH_SIZE]
               for i in range(0, len(needs_embedding), EMBEDDING_BATCH_SIZE)]

    print(f'Processing {len(batches)} batches of up to {EMBEDDING_BATCH_SIZE}...')

    for batch_num, batch in enumerate(batches):
        texts = [c['combined'] for c in batch]

        response = client.embeddings.create(
            model='text-embedding-3-small',  # See note above re: production vs. notebook model
            input=texts
        )

        for j, item in enumerate(response.data):
            batch[j]['embedding'] = item.embedding
            embedding_progress[batch[j]['text_id']] = item.embedding

        # Save progress after each batch
        with open(EMBEDDING_PROGRESS_FILE, 'w') as f:
            json.dump(embedding_progress, f)

        print(f'  Batch {batch_num+1}/{len(batches)} complete ({len(embedding_progress)} total)')
        time.sleep(EMBEDDING_DELAY_SECONDS)

embedded_count = sum(1 for c in all_chunks if c['embedding'] and c['embedding'] != [0.0]*1536)
print(f'\nEmbeddings complete. {embedded_count} chunks with real embeddings.')

Chunks needing embeddings: 886
Processing 9 batches of up to 100...
  Batch 1/9 complete (100 total)
  Batch 2/9 complete (200 total)
  Batch 3/9 complete (300 total)
  Batch 4/9 complete (400 total)
  Batch 5/9 complete (500 total)
  Batch 6/9 complete (600 total)
  Batch 7/9 complete (700 total)
  Batch 8/9 complete (800 total)
  Batch 9/9 complete (886 total)

Embeddings complete. 886 chunks with real embeddings.


In [18]:
# Cell 10: Export JSON Corpus
#
# Produces lincoln_speech_corpus.json
# Format: list of dicts, one per chunk, matching the existing schema
# with the new fields (keywords_semantic, chunk_context, position, etc.) added.

# Fields to include in the JSON (matches app's expected schema + new fields)
JSON_FIELDS = ['text_id', 'source', 'full_text', 'summary', 'keywords',
               'keywords_semantic', 'chunk_context', 'position', 'total_in_source',
               'source_file', 'source_type']

json_output = [{k: c[k] for k in JSON_FIELDS} for c in all_chunks]

with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(json_output, f, ensure_ascii=False, indent=2)

print(f'Exported {len(json_output)} chunks to {OUTPUT_JSON}')
print(f'File size: {os.path.getsize(OUTPUT_JSON) / 1024:.1f} KB')

# Verify: check a sample entry
sample = json_output[0]
print(f'\nSample chunk (first entry):')
print(f'  text_id: {sample["text_id"]}')
print(f'  source: {sample["source"]}')
print(f'  chunk_context: {sample["chunk_context"]}')
print(f'  full_text length: {len(sample["full_text"])} chars')
print(f'  summary: {sample["summary"][:100]}...')
print(f'  keywords: {sample["keywords"][:100]}')

Exported 886 chunks to /content/lincoln_speech_corpus_new.json
File size: 1728.6 KB

Sample chunk (first entry):
  text_id: Text #: 0
  source: Source:  At Peoria, Illinois. October 16, 1854.
  chunk_context: From Lincoln's At Peoria, Illinois. October 16, 1854., passage 1 of 5.
  full_text length: 883 chars
  summary: In this passage, Lincoln argues that the repeal of the Missouri Compromise and the question of its r...
  keywords: ['repeal', 'Missouri', 'Compromise', 'propriety', 'restoration', 'merits', 'patriotism', 'motives', 'remarks', 'points', 'class', 'view', 'sufficient', 'differently', 'arise']


In [19]:
# Cell 11: Export Embedding CSV
#
# Produces lincoln_index_embedded.csv
# Matches the schema of the current production CSV but with:
#   - 'Unnamed: 0' column removed
#   - 'summary' field without trailing space bug
#   - New schema fields added
#   - Embeddings stored as JSON string per row (matching current format)

import ast

# Build DataFrame
rows = []
for chunk in all_chunks:
    rows.append({
        'text_id': chunk['text_id'],
        'source': chunk['source'],
        'summary': chunk['summary'],    # Clean field name, no trailing space
        # Serialize keywords list as JSON string for CSV storage
        'keywords': json.dumps(chunk['keywords']) if isinstance(chunk['keywords'], list) else chunk['keywords'],
        'keywords_semantic': chunk['keywords_semantic'],
        'chunk_context': chunk['chunk_context'],
        'position': chunk['position'],
        'total_in_source': chunk['total_in_source'],
        'source_file': chunk['source_file'],
        'source_type': chunk['source_type'],
        'combined': chunk['combined'],
        'embedding': json.dumps(chunk['embedding']),  # Store as JSON string
    })

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)

print(f'Exported {len(df)} rows to {OUTPUT_CSV}')
print(f'File size: {os.path.getsize(OUTPUT_CSV) / 1024:.1f} KB')
print(f'Columns: {list(df.columns)}')

Exported 886 rows to /content/lincoln_index_embedded.csv
File size: 31385.8 KB
Columns: ['text_id', 'source', 'summary', 'keywords', 'keywords_semantic', 'chunk_context', 'position', 'total_in_source', 'source_file', 'source_type', 'combined', 'embedding']


In [20]:
# Cell 12: Validation and Summary Report

print('=== FINAL VALIDATION REPORT ===')
print()

# Re-load outputs to verify round-trip integrity
with open(OUTPUT_JSON, 'r') as f:
    loaded_json = json.load(f)

loaded_csv = pd.read_csv(OUTPUT_CSV)

print(f'JSON: {len(loaded_json)} chunks')
print(f'CSV:  {len(loaded_csv)} rows')
assert len(loaded_json) == len(loaded_csv), 'JSON and CSV counts do not match!'
print('✓ JSON and CSV row counts match')

# Check no empty full_text
empty_text = [c['text_id'] for c in loaded_json if not c['full_text']]
if empty_text:
    print(f'WARNING: {len(empty_text)} chunks with empty full_text: {empty_text[:5]}')
else:
    print('✓ No chunks with empty full_text')

# Check metadata completeness (will show DRY RUN placeholders if not real)
empty_summary = sum(1 for c in loaded_json if not c['summary'] or c['summary'] == '[DRY RUN SUMMARY]')
print(f'Chunks with real summaries: {len(loaded_json) - empty_summary}/{len(loaded_json)}')

# Check embeddings in CSV
has_embedding = loaded_csv['embedding'].apply(lambda x: len(json.loads(x)) == 1536 if pd.notna(x) else False)
real_embeddings = has_embedding.sum()
print(f'Chunks with real embeddings (1536-dim): {real_embeddings}/{len(loaded_csv)}')

# Source type breakdown
mc_count = sum(1 for c in loaded_json if c['source_type'] == 'miller_center')
ld_count = sum(1 for c in loaded_json if c['source_type'] == 'lincoln_douglas')
print(f'\nSource breakdown:')
print(f'  Miller Center:          {mc_count} chunks')
print(f'  Lincoln-Douglas Debates:{ld_count} chunks')
print(f'  Total:                  {len(loaded_json)} chunks')

# Size stats
sizes = [len(c['full_text']) for c in loaded_json]
print(f'\nChunk size stats:')
print(f'  Mean: {sum(sizes)/len(sizes):.0f} chars')
print(f'  Min:  {min(sizes)} chars')
print(f'  Max:  {max(sizes)} chars')
print(f'  Over 1500 chars: {sum(1 for s in sizes if s > 1500)}')

print()
print('=== DONE ===')
print(f'Outputs: {OUTPUT_JSON}, {OUTPUT_CSV}')

=== FINAL VALIDATION REPORT ===

JSON: 886 chunks
CSV:  886 rows
✓ JSON and CSV row counts match
✓ No chunks with empty full_text
Chunks with real summaries: 886/886
Chunks with real embeddings (1536-dim): 886/886

Source breakdown:
  Miller Center:          487 chunks
  Lincoln-Douglas Debates:399 chunks
  Total:                  886 chunks

Chunk size stats:
  Mean: 889 chars
  Min:  68 chars
  Max:  1863 chars
  Over 1500 chars: 5

=== DONE ===
Outputs: /content/lincoln_speech_corpus_new.json, /content/lincoln_index_embedded.csv
